In [ ]:
import json
import pandas as pd
from pathlib import Path


def load_jsonl(path):
    with open(path, encoding='utf8') as f:
        return [json.loads(l) for l in f]

# docs  = load_jsonl('../output/documents_enriched_02.jsonl', )
# qrels = load_jsonl('../output/qrels_enriched_02.jsonl')
docs  = load_jsonl('../output/documents.jsonl', )
qrels = load_jsonl('../output/qrels.jsonl')

print(f'Documents : {len(docs)}')
print(f'Qrels     : {len(qrels)}')

In [ ]:
print(f"Columns docs: {list(docs[0].keys())}")
print(f"Columns qrels: {list(qrels[0].keys())}")

Columns docs: ['id', 'source_dataset', 'source_type', 'title', 'main_text', 'secondary_texts', 'structured_fields', 'entities', 'relations', 'retrieval_metadata', 'raw_source']
Columns qrels: ['query_id', 'candidate_ids', 'relation_type']


In [ ]:
# High-level overview as a flat dataframe
df = pd.json_normalize(docs)
df[['id','source_type','source_dataset',
    'retrieval_metadata.is_queryable',
    'retrieval_metadata.is_candidate']].value_counts('source_type')

source_type
paper           64183
github_issue    27232
query             597
Name: count, dtype: int64

In [ ]:
# Inspect one record per source_type
for source_type in ['github_issue', 'paper', 'query']:
    ex = next(d for d in docs if d['source_type'] == source_type)
    print(f'\n── {source_type} ──')
    print(json.dumps(ex, indent=2))


── github_issue ──
{
  "id": "0",
  "source_dataset": "05nelsonm/TorOnionProxyLibrary-Android",
  "source_type": "github_issue",
  "title": "App crashing sometimes when ServiceAction.RESTART is being executed.",
  "main_text": "# Description\r\n<!-- Short description of the issue  -->\r\nApp is crashing after Tor is restarted. It reaches 95% Bootstrapped, stalls, then crashes.\r\n\r\n## Steps To Reproduce\r\n<!--- Provide a link to a live example, or an unambiguous set of steps to -->\r\n<!--- reproduce this bug. Include code to reproduce, if relevant          -->\r\nHappens sporadically, and unsure if it's only something with sending the `ServiceAction` via the notification's `PendingIntent`, or if it's also happening with the `TorServiceController.restartTor` method, too.",
  "secondary_texts": [
    "Stack trace:\r\n```\r\n2020-06-28 06:05:32.947 20706-21136/io.matthewnelson.sampleapp E/AndroidRuntime: FATAL EXCEPTION: Thread-5\r\n    Process: io.matthewnelson.sampleapp, PID: 20706

In [ ]:
# Qrels overview
pd.DataFrame(qrels)['relation_type'].value_counts()

relation_type
linked_issue      2934
corpusid_match     597
Name: count, dtype: int64

In [ ]:
docs_by_id = {str(d["id"]): d for d in docs}

for q in qrels[:3]:
    query_id = str(q["query_id"])
    print("query_id:", query_id)
    print("candidate_ids:", q["candidate_ids"])
    print("relation_type:", q.get("relation_type"))

    query_doc = docs_by_id[query_id]
    print("query title:", query_doc.get("title"))
    print("query desc:", query_doc.get("description", "").replace("\n", " ")[:180])

    for cid in q["candidate_ids"]:
        cand_doc = docs_by_id[str(cid)]
        print("  cand_id:", cid)
        print("  cand title:", cand_doc.get("title"))
        print("  cand desc:", cand_doc.get("main_text", "").replace("\n", " ")[:180])

    print("-" * 80)

query_id: 76
candidate_ids: ['77', '79']
relation_type: linked_issue
query title: Reduce amount of fields in DefaultPublisher
query desc: 
  cand_id: 77
  cand title: Improve fallback logic in resolution policy's resolve method
  cand desc: `override fun resolve(request: TrackableResolutionRequest): Resolution `  If the intention was to implement the logic from here: https://github.com/ably/ably-asset-tracking-android
  cand_id: 79
  cand title: Remember last sent locations for each of the Trackables independently
  cand desc: We have two fields for locations `lastSentRaw` and `lastSentEnhanced` and they're holding the value of the last location that was received from the location engine. Those values ar
--------------------------------------------------------------------------------
query_id: 77
candidate_ids: ['76']
relation_type: linked_issue
query title: Improve fallback logic in resolution policy's resolve method
query desc: 
  cand_id: 76
  cand title: Reduce amount of fields in 

In [ ]:
# That is the maximum of candidate ids a query has. This should give us a feeling which k might be needed.
ground_truth_count = [len(q["candidate_ids"]) for q in qrels]
print(f"max_k: {max(ground_truth_count)}")

# Average
print(f"avg k : {sum(ground_truth_count) / len(ground_truth_count)}")

max_k: 14
avg k : 1.7063154913622203


In [ ]:
print(f"Non empty sec texts: {sum(1 for d in docs if d["secondary_texts"])}")
print(f"Non empty sec texts paper: {sum(1 for d in docs if d["secondary_texts"] and d["source_type"] == "paper")}")
print(f"Non empty sec texts gh_issue: {sum(1 for d in docs if d["secondary_texts"] and d["source_type"] == "github_issue")}")

Non empty sec texts: 91405
Non empty sec texts paper: 64173
Non empty sec texts gh_issue: 27232


In [ ]:
[d["structured_fields"] for d in docs]

[{'categorical': 'bug', 'hierarchical': {}},
 {'categorical': 'enhancement', 'hierarchical': {}},
 {'categorical': {}, 'hierarchical': {}},
 {'categorical': {}, 'hierarchical': {}},
 {'categorical': 'enhancement', 'hierarchical': {}},
 {'categorical': {}, 'hierarchical': {}},
 {'categorical': {}, 'hierarchical': {}},
 {'categorical': {}, 'hierarchical': {}},
 {'categorical': 'enhancement', 'hierarchical': {}},
 {'categorical': 'layer:backend', 'hierarchical': {}},
 {'categorical': 'layer:backend', 'hierarchical': {}},
 {'categorical': 'layer:backend', 'hierarchical': {}},
 {'categorical': {}, 'hierarchical': {}},
 {'categorical': 'layer:backend', 'hierarchical': {}},
 {'categorical': {}, 'hierarchical': {}},
 {'categorical': {}, 'hierarchical': {}},
 {'categorical': 'layer:backend', 'hierarchical': {}},
 {'categorical': 'layer:backend', 'hierarchical': {}},
 {'categorical': {}, 'hierarchical': {}},
 {'categorical': 'enhancement', 'hierarchical': {}},
 {'categorical': 'priority:critical